In [9]:
# --- Cell 1: imports ---
import sys
import numpy as np
import open_clip
import torch
sys.path.append('..')
sys.path.append('../..')

from common.seed import set_seed
from common.io_utils import save_results, load_results
from backbones.backbones import get_resnet50, get_vit, get_clip, extract_all_features
from heads.train_head import train_linear_head
from evaluations.evaluate_bias import evaluate_head, clip_zero_shot_eval, evaluate_head_with_preds
from img_prep.make_subset import load_stl10




set_seed(6304)

In [10]:
# --- Cell 2: load the SAME subset from the saved file ---
subset_ids = load_results("../results/subset_ids.json")
train_idx = subset_ids["train_indices"]
val_idx = subset_ids["val_indices"]
test_subset_idx = subset_ids["test_subset_indices"]

train_ds = load_stl10(split='train')
test_ds = load_stl10(split='test')

class_names = ['airplane', 'bird', 'car', 'cat', 'deer', 'dog', 'horse', 'monkey', 'ship', 'truck']

In [11]:
# --- Cell 3: load all 3 backbones ---
from img_prep.transforms import resnet_normalize, vit_normalize, clip_normalize

resnet, _ = get_resnet50()
vit, _ = get_vit()
clip_model, _ = get_clip()
tokenizer = open_clip.get_tokenizer('ViT-B-32')

c:\Users\javai\anaconda3\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [12]:
# --- Cell 4: extract + cache features for each backbone, once ---
def get_or_cache_features(model, dataset, indices, transform, model_type, name, split):
    cache_path = f"../../datasets/cache/{name}_{split}_features.npz"
    try:
        data = np.load(cache_path)
        return data['features'], data['labels']
    except FileNotFoundError:
        feats, labels = extract_all_features(model, dataset, indices, transform, model_type=model_type)
        np.savez(cache_path, features=feats, labels=labels)
        return feats, labels

backbones_info = [
    ("resnet50", resnet, resnet_normalize, "resnet_or_vit"),
    ("vit_b16", vit, vit_normalize, "resnet_or_vit"),
    ("clip", clip_model, clip_normalize, "clip"),
]

features = {}
for name, model, tf, mtype in backbones_info:
    train_feats, train_labels = get_or_cache_features(model, train_ds, train_idx, tf, mtype, name, "train")
    val_feats, val_labels = get_or_cache_features(model, train_ds, val_idx, tf, mtype, name, "val")
    test_feats, test_labels = get_or_cache_features(model, test_ds, test_subset_idx, tf, mtype, name, "test")
    features[name] = {
        "train": (train_feats, train_labels),
        "val": (val_feats, val_labels),
        "test": (test_feats, test_labels)
    }
    print(f"{name}: train {train_feats.shape}, val {val_feats.shape}, test {test_feats.shape}")

resnet50: train (4000, 2048), val (1000, 2048), test (500, 2048)
vit_b16: train (4000, 768), val (1000, 768), test (500, 768)
clip: train (4000, 512), val (1000, 512), test (500, 512)


In [13]:
# --- Cell 5: train a linear head for ResNet and ViT ---
results = {"step": "clean_baseline", "seed": 6304, "models": {}}
clean_predictions = {}

for name in ["resnet50", "vit_b16"]:
    train_feats, train_labels = features[name]["train"]
    val_feats, val_labels = features[name]["val"]
    test_feats, test_labels = features[name]["test"]

    head = train_linear_head(train_feats, train_labels, val_feats, val_labels, seed=6304)
    metrics, preds = evaluate_head_with_preds(head, test_feats, test_labels)
    results["models"][name] = metrics
    clean_predictions[name] = preds.tolist()
    torch.save(head.state_dict(), f"../../datasets/cache/{name}_head.pth")
    print(name, metrics)

resnet50 {'top1_acc': 0.952, 'macro_f1': 0.9519785680898508, 'mean_max_conf': 0.7020431160926819}
vit_b16 {'top1_acc': 0.964, 'macro_f1': 0.9641221323159218, 'mean_max_conf': 0.8672109246253967}


In [14]:
# --- Cell 6: CLIP — train a head on its features ---
train_feats, train_labels = features["clip"]["train"]
val_feats, val_labels = features["clip"]["val"]
test_feats, test_labels = features["clip"]["test"]

clip_head = train_linear_head(train_feats, train_labels, val_feats, val_labels, seed=6304)
clip_head_metrics, clip_head_preds = evaluate_head_with_preds(clip_head, test_feats, test_labels)
results["models"]["clip_head"] = clip_head_metrics
clean_predictions["clip_head"] = clip_head_preds.tolist()
torch.save(clip_head.state_dict(), "../../datasets/cache/clip_head_head.pth")
print("clip_head", clip_head_metrics)

clip_head {'top1_acc': 0.956, 'macro_f1': 0.955741066202758, 'mean_max_conf': 0.12711185216903687}


In [15]:
# --- Cell 7: CLIP zero-shot ---
from torch.utils.data import Subset
from img_prep.transforms import to_common_224, clip_normalize

test_subset = Subset(test_ds, test_subset_idx)
imgs = torch.stack([clip_normalize(to_common_224(img)) for img, _ in test_subset])
labels = np.array([lbl for _, lbl in test_subset])

zeroshot_metrics, zeroshot_preds = clip_zero_shot_eval(clip_model, tokenizer, imgs, labels, class_names)
results["models"]["clip_zeroshot"] = zeroshot_metrics
clean_predictions["clip_zeroshot"] = zeroshot_preds.tolist()
print("clip_zeroshot", zeroshot_metrics)

clip_zeroshot {'top1_acc': 0.934, 'macro_f1': 0.9327825612360451, 'mean_max_conf': 0.9264334440231323}


In [16]:
# --- Cell 8: save everything ---
save_results(results, "../results/clean_baseline.json")
save_results({"seed": 6304, "predictions": clean_predictions}, "../results/clean_predictions.json")
print(results)

{'step': 'clean_baseline', 'seed': 6304, 'models': {'resnet50': {'top1_acc': 0.952, 'macro_f1': 0.9519785680898508, 'mean_max_conf': 0.7020431160926819}, 'vit_b16': {'top1_acc': 0.964, 'macro_f1': 0.9641221323159218, 'mean_max_conf': 0.8672109246253967}, 'clip_head': {'top1_acc': 0.956, 'macro_f1': 0.955741066202758, 'mean_max_conf': 0.12711185216903687}, 'clip_zeroshot': {'top1_acc': 0.934, 'macro_f1': 0.9327825612360451, 'mean_max_conf': 0.9264334440231323}}}
